In [ ]:
import pandas as pd
import numpy as np
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import LeaveOneOut

In [2]:
path= r'C:\Users\homel\Downloads\mnist_train.csv'
df= pd.read_csv(path)

In [3]:
df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
# Separate the features and labels
X = df.iloc[:, 1:].values
y = df['label'].values

In [5]:
# Define the logistic regression objective function
def F(beta, X, y, lam):
    m = len(y)
    h = 1 / (1 + np.exp(-X.dot(beta)))
    return -(1/m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h)) + (lam/2) * np.sum(beta**2)

In [6]:
# Compute the gradient of the objective function
def grad_F(beta, X, y, lam):
    m = len(y)
    h = 1 / (1 + np.exp(-X.dot(beta)))
    return -(1/m) * X.T.dot(y - h) + lam * beta

In [7]:
# Gradient descent algorithm
def gradient_descent(X, y, lam, alpha=0.01, tol=1e-6, max_iter=1000):
    m, n = X.shape
    beta = np.zeros(n)
    
    for i in range(max_iter):
        grad = grad_F(beta, X, y, lam)
        if np.linalg.norm(grad) < tol:
            break
        
        # Backtracking line search with Armijo-Goldstein condition
        c = 0.5
        alpha_k = alpha
        while F(beta - alpha_k * grad, X, y, lam) > F(beta, X, y, lam) - c * alpha_k * np.linalg.norm(grad)**2:
            alpha_k *= 0.5
        
        beta = beta - alpha_k * grad
    
    return beta

In [ ]:
# Use leave-one-out cross-validation to tune the regularization parameter λ
loo = LeaveOneOut()
best_lambda = None
best_score = 0
for lam in [0.01, 0.1, 1, 10, 100]:
    scores = []
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        beta = gradient_descent(X_train, y_train, lam)
        y_pred = 1 / (1 + np.exp(-X_test.dot(beta)))
        score = np.mean(y_test == (y_pred > 0.5))
        scores.append(score)
    
    mean_score = np.mean(scores)
    if mean_score > best_score:
        best_lambda = lam
        best_score = mean_score

C:\Users\homel\AppData\Local\Temp\ipykernel_17000\3861204306.py:5: RuntimeWarning: divide by zero encountered in log
  return -(1/m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h)) + (lam/2) * np.sum(beta**2)
C:\Users\homel\AppData\Local\Temp\ipykernel_17000\3861204306.py:5: RuntimeWarning: invalid value encountered in multiply
  return -(1/m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h)) + (lam/2) * np.sum(beta**2)
C:\Users\homel\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:88: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [ ]:
print(f"Best regularization parameter (λ): {best_lambda}")
print(f"Best cross-validation score: {best_score}")

In [ ]:
# Load the test data
path2 = r'C:\Users\homel\Downloads\mnist_test.csv'
df2 = pd.read_csv(path2)


In [ ]:
df2.head()

In [ ]:
# Split the data into features and target variable
X = df.iloc[:, 1:]  # Features are the columns after the 'label' column
y = df['label']    # Target variable is the 'label' column

In [ ]:
# Create and train the logistic regression model
model = LogisticRegression()
model.fit(X, y)

In [ ]:
# Make predictions on the new data
y_pred = model.predict(X)

In [ ]:
# Evaluate the model's performance
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y, y_pred)
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
conf_matrix = confusion_matrix(y, y_pred)
report = classification_report(y, y_pred)

In [ ]:
# Visualization 1: Algorithm
plt.subplot(1, 2, 1)
plt.plot(range(len(model.coef_[0])), model.coef_[0], label='Coefficients')
plt.xlabel('Feature')
plt.ylabel('Coefficient Value')
plt.title('Logistic Regression Coefficients')
plt.legend()

In [ ]:
# Visualization 2: Fitness
plt.subplot(1, 2, 2)
plt.plot(range(len(model.intercept_)), model.intercept_, label='Intercept')
plt.xlabel('Iteration')
plt.ylabel('Intercept Value')
plt.title('Logistic Regression Intercept')
plt.legend()